# Clase 193 — Stack bayesiano moderno: PyMC v5 + NumPyro + ArviZ

PyMC v5 (PyTensor backend), NumPyro (JAX, NUTS rápido), ArviZ (diagnósticos y comparación de modelos).
Requiere: `pip install numpy scipy arviz` (opcional `pymc`, `numpyro`).

In [ ]:
import numpy as np
rng = np.random.default_rng(42)
n = 200
TRUE_A, TRUE_B, TRUE_SIGMA = 1.5, 2.3, 1.0
x = rng.normal(0, 1, n)
y = TRUE_A + TRUE_B * x + rng.normal(0, TRUE_SIGMA, n)
print(f'n={n}  truth: a={TRUE_A}  b={TRUE_B}  sigma={TRUE_SIGMA}')

## Modelo bayesiano
$$ a \sim N(0,10),\;\; b \sim N(0,10),\;\; \sigma \sim \text{HalfNormal}(5),\;\; y_i \sim N(a + b x_i, \sigma) $$

## Opción A — PyMC v5

In [ ]:
PYMC_OK = False
try:
    import pymc as pm
    import arviz as az
    with pm.Model() as model:
        a = pm.Normal('a', 0, 10)
        b = pm.Normal('b', 0, 10)
        sigma = pm.HalfNormal('sigma', 5)
        mu = a + b * x
        pm.Normal('y_obs', mu=mu, sigma=sigma, observed=y)
        idata = pm.sample(1000, tune=1000, chains=2, random_seed=42, progressbar=False)
    PYMC_OK = True
    print(az.summary(idata, var_names=['a','b','sigma'], round_to=3))
except Exception as e:
    print(f'PyMC no disponible ({type(e).__name__}); fallback al Metropolis manual abajo.')

## ArviZ — diagnósticos y plots

In [ ]:
if PYMC_OK:
    import matplotlib.pyplot as plt
    az.plot_trace(idata, var_names=['a','b','sigma']); plt.tight_layout(); plt.show()
    az.plot_posterior(idata, var_names=['a','b','sigma'], hdi_prob=0.94); plt.show()
    # rhat / ess
    rhat = az.rhat(idata)
    ess = az.ess(idata)
    print('rhat:', {k: float(v) for k,v in rhat.items() if k in ['a','b','sigma']})
    print('ess :', {k: float(v) for k,v in ess.items()  if k in ['a','b','sigma']})
    print('Regla: rhat < 1.01 y ess > 400/chain -> OK')
else:
    print('Skip ArviZ plots (PyMC no disponible).')

## Fallback — Metropolis-Hastings manual
Si PyMC no está, esto da una posterior cruda para verificar la lógica.

In [ ]:
from scipy.stats import norm, halfnorm

def log_post(a, b, sigma, x, y):
    if sigma <= 0: return -np.inf
    lp  = norm.logpdf(a, 0, 10) + norm.logpdf(b, 0, 10) + halfnorm.logpdf(sigma, scale=5)
    lp += norm.logpdf(y, loc=a + b*x, scale=sigma).sum()
    return lp

def metropolis(x, y, n_iter=8000, burn=2000, seed=42):
    r = np.random.default_rng(seed)
    a, b, s = 0.0, 0.0, 1.0
    chain = np.empty((n_iter, 3))
    cur = log_post(a, b, s, x, y)
    for i in range(n_iter):
        ap, bp, sp = a + r.normal(0, 0.15), b + r.normal(0, 0.15), s + r.normal(0, 0.1)
        prop = log_post(ap, bp, sp, x, y)
        if np.log(r.uniform()) < prop - cur:
            a, b, s, cur = ap, bp, sp, prop
        chain[i] = a, b, s
    return chain[burn:]

ch = metropolis(x, y)
print('Metropolis manual:')
print(f'  a    : mean={ch[:,0].mean():.3f}  HDI94=[{np.percentile(ch[:,0],3):.3f}, {np.percentile(ch[:,0],97):.3f}]')
print(f'  b    : mean={ch[:,1].mean():.3f}  HDI94=[{np.percentile(ch[:,1],3):.3f}, {np.percentile(ch[:,1],97):.3f}]')
print(f'  sigma: mean={ch[:,2].mean():.3f}  HDI94=[{np.percentile(ch[:,2],3):.3f}, {np.percentile(ch[:,2],97):.3f}]')

## Opción B — NumPyro (JAX, NUTS rápido)

In [ ]:
try:
    import numpyro
    import numpyro.distributions as dist
    from numpyro.infer import MCMC, NUTS
    import jax.numpy as jnp
    from jax import random as jrandom

    def model_np(x, y=None):
        a = numpyro.sample('a', dist.Normal(0, 10))
        b = numpyro.sample('b', dist.Normal(0, 10))
        sigma = numpyro.sample('sigma', dist.HalfNormal(5))
        numpyro.sample('y_obs', dist.Normal(a + b*x, sigma), obs=y)

    kernel = NUTS(model_np)
    mcmc = MCMC(kernel, num_warmup=500, num_samples=1000, num_chains=2, progress_bar=False)
    mcmc.run(jrandom.PRNGKey(42), x=jnp.array(x), y=jnp.array(y))
    mcmc.print_summary()
except ImportError:
    print('numpyro no instalado; `pip install numpyro` para NUTS sobre JAX.')

## Comparación de modelos con ArviZ (WAIC / LOO)
Si tenemos un modelo M1 (lineal) y M2 (cuadrático), ¿cuál predice mejor out-of-sample?

In [ ]:
if PYMC_OK:
    # Modelo cuadratico
    with pm.Model() as model2:
        a = pm.Normal('a', 0, 10)
        b = pm.Normal('b', 0, 10)
        c = pm.Normal('c', 0, 10)
        sigma = pm.HalfNormal('sigma', 5)
        mu = a + b*x + c*x**2
        pm.Normal('y_obs', mu=mu, sigma=sigma, observed=y)
        idata2 = pm.sample(1000, tune=1000, chains=2, random_seed=42, progressbar=False, idata_kwargs={'log_likelihood': True})
    # Re-sample del modelo 1 con log_lik
    with model:
        pm.compute_log_likelihood(idata)
    cmp = az.compare({'linear': idata, 'cuadratico': idata2}, ic='loo')
    print(cmp)
    print('\nMejor = el de arriba (loo mas alto = mejor predictivo).')
else:
    print('Skip comparación (PyMC no disponible).')

## Diagnósticos clave
- **R-hat < 1.01**: cadenas convergieron.
- **ESS > 400/cadena**: suficiente información independiente.
- **Divergences = 0** (NUTS): si hay, reparametrizar o subir `target_accept`.
- **WAIC/LOO**: comparar predictivo entre modelos.

## Takeaways
1. **PyMC v5** = Python idiomatic, backend PyTensor, NUTS por default.
2. **NumPyro** = JAX → orden de magnitud más rápido en CPU/GPU, API similar.
3. **ArviZ** = lingua franca de inferencia bayesiana (idata, plots, comparación).
4. Workflow: definir modelo → sample → revisar rhat/ess/divergences → posterior plots → comparar con LOO.
5. Si nada de eso está disponible, Metropolis manual hace el trabajo conceptual — sólo no escalá.

## ✅ Soluciones de los ejercicios

Soluciones trabajadas y **ejecutables** de los ejercicios del README (sección `## 🧪 Ejercicios`). Datos sintéticos con `np.random.default_rng(42)`, sin internet. Cada bloque imprime resultados y valida con `assert`.

> **Nota (stack bayesiano):** PyMC/NumPyro/ArviZ no están instalados. Los bloques con esas librerías son **código correcto ilustrativo** (en `try/except`); la lógica se valida con un **Gibbs sampler conjugado en numpy/scipy** que corre con `assert`.

### Ejercicio 1 — Jerárquico en PyMC v5 (+ Gibbs ejecutable)
`tip ~ Normal(α_day + β·bill, σ)`, `α_day ~ Normal(μα, σα)`. Código PyMC ilustrativo + Gibbs conjugado ejecutable que recupera los parámetros.

In [ ]:
# --- PyMC v5 jerarquico (ilustrativo) ---
try:
    import pymc as pm
    # (datos xg, group, yg se definen en la celda ejecutable siguiente)
    print("PyMC disponible: usar pm.Model() con alpha_day ~ Normal(mu_a, sigma_a).")
except Exception as e:
    print(f"PyMC no disponible ({type(e).__name__}); Gibbs conjugado ejecutable abajo.")

In [ ]:
# --- Gibbs sampler conjugado (ejecutable) para el modelo jerarquico gaussiano ---
import numpy as np
from scipy import stats
rng = np.random.default_rng(42)
G, n_g = 4, 60                                  # 4 grupos (dias)
mu_a_true, sigma_a_true, beta_true, sigma_true = 5.0, 2.0, 1.5, 1.0
alpha_g_true = rng.normal(mu_a_true, sigma_a_true, G)
group = np.repeat(np.arange(G), n_g)
xg = rng.normal(0, 1, G*n_g)
yg = alpha_g_true[group] + beta_true*xg + rng.normal(0, sigma_true, G*n_g)

def gibbs_hier(n_iter=4000, burn=1000, seed=0):
    r = np.random.default_rng(seed)
    a = np.zeros(G); b = 0.0; sig2 = 1.0; sig2_a = 1.0; mu_a = 0.0
    tau_b = tau_mu = 100.0
    idx = [np.where(group == j)[0] for j in range(G)]
    nj = np.array([len(i) for i in idx]); sx2 = (xg**2).sum()
    keep = []
    for it in range(n_iter):
        for j in range(G):                       # alpha_j | resto (conjugado normal)
            rj = (yg[idx[j]] - b*xg[idx[j]]).sum()
            prec = nj[j]/sig2 + 1/sig2_a
            a[j] = r.normal((rj/sig2 + mu_a/sig2_a)/prec, np.sqrt(1/prec))
        res = yg - a[group]                        # beta | resto
        prec = sx2/sig2 + 1/tau_b
        b = r.normal((res @ xg / sig2)/prec, np.sqrt(1/prec))
        prec = G/sig2_a + 1/tau_mu                 # mu_a | resto
        mu_a = r.normal((a.sum()/sig2_a)/prec, np.sqrt(1/prec))
        resid = yg - a[group] - b*xg               # sigma^2 | resto (inverse-gamma)
        sig2 = 1/r.gamma(2 + len(yg)/2, 1/(2 + 0.5*(resid**2).sum()))
        sig2_a = 1/r.gamma(2 + G/2, 1/(2 + 0.5*((a - mu_a)**2).sum()))
        if it >= burn:
            keep.append(np.concatenate([a, [b, np.sqrt(sig2), mu_a, np.sqrt(sig2_a)]]))
    return np.array(keep)

samples = gibbs_hier()
a_hat = samples[:, :G].mean(0); b_hat = samples[:, G].mean()
hdi_b = np.percentile(samples[:, G], [3, 97])
print("alpha por grupo (post): ", np.round(a_hat, 2), " verdadero:", np.round(alpha_g_true, 2))
print(f"beta = {b_hat:.3f}  HDI94% = ({hdi_b[0]:.3f}, {hdi_b[1]:.3f})  (verdadero {beta_true})")
assert abs(b_hat - beta_true) < 0.3 and np.mean(np.abs(a_hat - alpha_g_true)) < 1.0

### Ejercicio 2 — Non-centered parameterization
`α_j = μα + σα·z_j`, `z_j ~ N(0,1)`. Desacopla nivel y escala y evita el *funnel* que genera divergencias en NUTS.

In [ ]:
rng = np.random.default_rng(1)
mu, sigma = 2.0, 0.7
z = rng.normal(0, 1, 100_000)
x_nc = mu + sigma*z                               # non-centered
x_c = rng.normal(mu, sigma, 100_000)              # centered
print(f"centered    : mean={x_c.mean():.3f}  sd={x_c.std():.3f}")
print(f"non-centered: mean={x_nc.mean():.3f}  sd={x_nc.std():.3f}")
print("En jerarquicos: alpha_j = mu_a + sigma_a * z_j (z_j~N(0,1)) elimina divergencias del sampler.")
assert abs(x_nc.mean() - mu) < 0.02 and abs(x_nc.std() - sigma) < 0.02

### Ejercicio 3 — Posterior predictive check
Simular `y_rep` desde el posterior del Gibbs y comparar estadísticos con lo observado (bayesian p-value ≈ 0.5).

In [ ]:
S = samples.shape[0]
sel = rng.integers(0, S, 2000)
y_rep = np.empty((2000, len(yg)))
for k, s in enumerate(sel):
    a_s, b_s, sd_s = samples[s, :G], samples[s, G], samples[s, G+1]
    y_rep[k] = a_s[group] + b_s*xg + rng.normal(0, sd_s, len(yg))
p_mean = np.mean(y_rep.mean(1) >= yg.mean())
p_sd = np.mean(y_rep.std(1) >= yg.std())
print(f"bayesian p-value media = {p_mean:.2f}  sd = {p_sd:.2f}  (~0.5 = buen ajuste)")
assert 0.05 < p_mean < 0.95

### Ejercicio 4 — NumPyro
Mismo modelo jerárquico en **NumPyro** (JAX). Código correcto ilustrativo; corre en entorno con JAX.

In [ ]:
try:
    import numpyro, numpyro.distributions as dist
    from numpyro.infer import MCMC, NUTS
    import jax.numpy as jnp
    from jax import random as jr
    def hier(group, x, y=None, G=G):
        mu_a = numpyro.sample("mu_a", dist.Normal(0, 5))
        sigma_a = numpyro.sample("sigma_a", dist.HalfNormal(2))
        with numpyro.plate("g", G):
            a = numpyro.sample("a", dist.Normal(mu_a, sigma_a))
        b = numpyro.sample("b", dist.Normal(0, 1))
        s = numpyro.sample("sigma", dist.HalfNormal(2))
        numpyro.sample("y", dist.Normal(a[group] + b*x, s), obs=y)
    mcmc = MCMC(NUTS(hier), num_warmup=500, num_samples=1000, progress_bar=False)
    mcmc.run(jr.PRNGKey(0), group=jnp.array(group), x=jnp.array(xg), y=jnp.array(yg))
    mcmc.print_summary()
except Exception as e:
    print(f"NumPyro no disponible ({type(e).__name__}); codigo correcto para entorno con JAX.")

### Ejercicio 5 — Comparación de modelos (WAIC)
`az.compare` no disponible → WAIC a mano sobre log-verosimilitudes puntuales. Comparamos intercepto-solo, +slope y jerárquico: el jerárquico gana (menor WAIC).

In [ ]:
def gibbs_linear(Xd, y, n_iter=3000, burn=1000, seed=0):
    r = np.random.default_rng(seed); p = Xd.shape[1]
    sig2 = 1.0; tau = 100.0; XtX = Xd.T @ Xd; keep = []
    for it in range(n_iter):
        cov = np.linalg.inv(XtX/sig2 + np.eye(p)/tau)
        beta = r.multivariate_normal(cov @ (Xd.T @ y / sig2), cov)
        resid = y - Xd @ beta
        sig2 = 1/r.gamma(2 + len(y)/2, 1/(2 + 0.5*resid @ resid))
        if it >= burn: keep.append(np.concatenate([beta, [np.sqrt(sig2)]]))
    return np.array(keep)

def waic(loglik):                                  # loglik: (draws, n)
    lppd = np.log(np.mean(np.exp(loglik), axis=0)).sum()
    p_w = np.var(loglik, axis=0, ddof=1).sum()
    return -2*(lppd - p_w)                          # menor = mejor

def ll_linear(draws, Xd, y):
    return np.array([stats.norm.logpdf(y, Xd @ d[:-1], d[-1]) for d in draws])
def ll_hier(sm):
    return np.array([stats.norm.logpdf(yg, s[:G][group] + s[G]*xg, s[G+1]) for s in sm])

X1 = np.ones((len(yg), 1))
X2 = np.column_stack([np.ones(len(yg)), xg])
w_int = waic(ll_linear(gibbs_linear(X1, yg), X1, yg))
w_slope = waic(ll_linear(gibbs_linear(X2, yg), X2, yg))
w_hier = waic(ll_hier(samples))
print(f"WAIC intercepto-solo = {w_int:.1f}")
print(f"WAIC + slope         = {w_slope:.1f}")
print(f"WAIC jerarquico      = {w_hier:.1f}  <- menor = mejor predictivo")
assert w_hier < w_slope and w_hier < w_int